In [1]:
import numpy as np
import pandas as pd
import urllib.request

URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data'
urllib.request.urlretrieve(URL, '../../parkinsons.data')
df = pd.read_csv('../../parkinsons.data')
print(f'Dataset shape: {df.shape}')

Dataset shape: (195, 24)


In [4]:
import os
import urllib.request
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                              balanced_accuracy_score, roc_auc_score)

if not os.path.exists('../../parkinsons.data'):
    urllib.request.urlretrieve(
        'https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data',
        '../../parkinsons.data')
df = pd.read_csv('../../parkinsons.data')
df['subject_id'] = df['name'].str.extract(r'^(.*)_\d+$')

X = df.drop(columns=['name', 'status', 'subject_id']).reset_index(drop=True)
y = df['status'].values
groups = df['subject_id'].values
print(f"Загружено: {len(X)} записей, {len(set(groups))} пациентов")

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('select', SelectKBest(f_classif, k=12)),
    ('svm', SVC(probability=True, class_weight='balanced', random_state=42))
])

param_grid = {
    'svm__C': [0.01, 0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'svm__kernel': ['rbf', 'linear']
}

logo = LeaveOneGroupOut()
rec_true, rec_pred, rec_prob, rec_subject, fold_best_params = [], [], [], [], []

for fold_i, (train_idx, test_idx) in enumerate(logo.split(X, y, groups=groups), 1):
    X_train, y_train, groups_train = X.iloc[train_idx], y[train_idx], groups[train_idx]

    inner_cv = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=42)
    grid = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring='balanced_accuracy', n_jobs=-1)
    grid.fit(X_train, y_train, groups=groups_train)

    fold_best_params.append(grid.best_params_)
    rec_true.extend(y[test_idx])
    rec_pred.extend(grid.predict(X.iloc[test_idx]))
    rec_prob.extend(grid.predict_proba(X.iloc[test_idx])[:, 1])
    rec_subject.extend(groups[test_idx])
    print(f"Fold {fold_i}/32: best_params={grid.best_params_}")

rec_true, rec_pred, rec_prob = map(np.array, (rec_true, rec_pred, rec_prob))

print(f"\n=== Recording-level (pooled out-of-fold, n={len(rec_true)}) ===")
print(f"accuracy      : {accuracy_score(rec_true, rec_pred):.3f}")
print(f"f1            : {f1_score(rec_true, rec_pred):.3f}")
print(f"sensitivity   : {recall_score(rec_true, rec_pred):.3f}")
print(f"specificity   : {recall_score(rec_true, rec_pred, pos_label=0):.3f}")
print(f"balanced_acc  : {balanced_accuracy_score(rec_true, rec_pred):.3f}")
print(f"auc           : {roc_auc_score(rec_true, rec_prob):.3f}")

res = pd.DataFrame({'subject': rec_subject, 'true': rec_true, 'pred': rec_pred, 'prob': rec_prob})
subj = res.groupby('subject').agg(true=('true', 'first'),
                                   pred=('pred', lambda s: s.mode()[0]),
                                   prob=('prob', 'mean'))

print(f"\n=== Subject-level (majority vote, n={len(subj)}) ===")
print(f"accuracy      : {accuracy_score(subj['true'], subj['pred']):.3f}")
print(f"f1            : {f1_score(subj['true'], subj['pred']):.3f}")
print(f"sensitivity   : {recall_score(subj['true'], subj['pred']):.3f}")
print(f"specificity   : {recall_score(subj['true'], subj['pred'], pos_label=0):.3f}")
print(f"balanced_acc  : {balanced_accuracy_score(subj['true'], subj['pred']):.3f}")
print(f"auc           : {roc_auc_score(subj['true'], subj['prob']):.3f}")

params_df = pd.DataFrame(fold_best_params)
print("\n=== Частота выбора гиперпараметров по 32 LOSO-фолдам (для R2.3) ===")
for col in params_df.columns:
    print(f"\n{col}:"); print(params_df[col].value_counts())

subj.reset_index().rename(columns={'subject': 'subject_id', 'true': 'label'})[['subject_id','label','prob','pred']] \
    .to_csv('results_voice.csv', index=False)
print("\nСохранено: results_voice.csv (обновлено)")

Загружено: 195 записей, 32 пациентов
Fold 1/32: best_params={'svm__C': 10, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
Fold 2/32: best_params={'svm__C': 100, 'svm__gamma': 0.001, 'svm__kernel': 'rbf'}
Fold 3/32: best_params={'svm__C': 10, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
Fold 4/32: best_params={'svm__C': 10, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
Fold 5/32: best_params={'svm__C': 10, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
Fold 6/32: best_params={'svm__C': 0.1, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
Fold 7/32: best_params={'svm__C': 100, 'svm__gamma': 0.001, 'svm__kernel': 'rbf'}
Fold 8/32: best_params={'svm__C': 0.1, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
Fold 9/32: best_params={'svm__C': 0.1, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
Fold 10/32: best_params={'svm__C': 1, 'svm__gamma': 0.001, 'svm__kernel': 'rbf'}
Fold 11/32: best_params={'svm__C': 1, 'svm__gamma': 0.001, 'svm__kernel': 'rbf'}
Fold 12/32: best_params={'svm__C': 1, 'svm__gamma': 0.001, 'svm__kernel': 

In [7]:
subj.reset_index().rename(columns={'subject': 'subject_id', 'true': 'label'})[['subject_id','label','prob','pred']] \
    .to_csv('results_voice.csv', index=False)
print("Сохранено: results_voice.csv")

Сохранено: results_voice.csv
